# Job Salary Prediction MVP — EDA and Training

This notebook contains data inspection, exploratory data analysis, preprocessing, baseline models, Genetic Algorithm feature selection, final model training, evaluation and explainability.

In [ ]:
import os
import sys
sys.path.append("../src")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from preprocessing import (
    ALL_FEATURES,
    NUMERIC_FEATURES,
    CATEGORICAL_FEATURES,
    build_preprocessor,
    get_transformed_feature_names,
)
from genetic_feature_selection import GeneticFeatureSelector

DATA_PATH = "../data/job_salary_prediction_dataset.csv"
df = pd.read_csv(DATA_PATH)


## 1. Data loading and basic inspection

In [ ]:
print("Shape:", df.shape)
display(df.head())

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes)

print("\nMissing values:")
display(df.isna().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nNumeric descriptive statistics:")
display(df.describe())

print("\nUnique values for categorical columns:")
for col in CATEGORICAL_FEATURES:
    print(f"{col}: {sorted(df[col].unique().tolist())}")

## 2. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(df["salary"], bins=40, kde=True)
plt.title("Salary distribution")
plt.xlabel("Salary")
plt.ylabel("Count")
plt.show()

print("Explanation: this chart shows the overall distribution of the target variable. It helps detect skewness, outliers and the typical salary range.")

In [ ]:
plt.figure(figsize=(12, 5))
avg_salary = df.groupby("job_title")["salary"].mean().sort_values(ascending=False)
sns.barplot(x=avg_salary.index, y=avg_salary.values)
plt.title("Average salary by job title")
plt.xlabel("Job title")
plt.ylabel("Average salary")
plt.xticks(rotation=45, ha="right")
plt.show()

print("Explanation: job title is expected to be one of the strongest salary-related categorical factors.")

In [ ]:
plt.figure(figsize=(8, 5))
order = ["High School", "Diploma", "Bachelor", "Master", "PhD"]
avg_salary = df.groupby("education_level")["salary"].mean().reindex(order)
sns.barplot(x=avg_salary.index, y=avg_salary.values)
plt.title("Average salary by education level")
plt.xlabel("Education level")
plt.ylabel("Average salary")
plt.show()

print("Explanation: this chart checks whether higher formal education is associated with higher average salary.")

In [ ]:
plt.figure(figsize=(10, 5))
avg_salary = df.groupby("experience_years")["salary"].mean()
sns.lineplot(x=avg_salary.index, y=avg_salary.values, marker="o")
plt.title("Average salary by experience years")
plt.xlabel("Experience years")
plt.ylabel("Average salary")
plt.show()

print("Explanation: salary usually increases with experience, so this chart checks the experience-salary relationship.")

In [ ]:
for col in ["industry", "company_size", "location", "remote_work"]:
    plt.figure(figsize=(11, 5))
    avg_salary = df.groupby(col)["salary"].mean().sort_values(ascending=False)
    sns.barplot(x=avg_salary.index, y=avg_salary.values)
    plt.title(f"Average salary by {col}")
    plt.xlabel(col)
    plt.ylabel("Average salary")
    plt.xticks(rotation=45, ha="right")
    plt.show()
    print(f"Explanation: this chart compares salary differences across {col} categories.")

In [ ]:
plt.figure(figsize=(7, 5))
corr = df[["experience_years", "skills_count", "certifications", "salary"]].corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation heatmap for numeric features")
plt.show()

print("Explanation: the heatmap shows linear relationships between numeric features and salary.")

## 3. Preprocessing

In [ ]:
X = df.drop(columns=["salary"])
y = df["salary"]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)

preprocessor = build_preprocessor(ALL_FEATURES)
preprocessor

## 4. Baseline models

In [ ]:
def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "R2": r2_score(y_true, y_pred),
    }

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest Regressor": RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1,
        max_depth=22,
    ),
    "Gradient Boosting Regressor": GradientBoostingRegressor(
        n_estimators=100,
        random_state=42,
        max_depth=4,
    ),
    "MLP Regressor": MLPRegressor(
        hidden_layer_sizes=(64, 32),
        max_iter=80,
        random_state=42,
        early_stopping=True,
    ),
}

baseline_results = {}
baseline_pipelines = {}

for name, model in models.items():
    print("Training:", name)
    pipeline = Pipeline([
        ("preprocessor", build_preprocessor(ALL_FEATURES)),
        ("model", model),
    ])
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)
    baseline_results[name] = regression_metrics(y_test, preds)
    baseline_pipelines[name] = pipeline

baseline_table = pd.DataFrame(baseline_results).T.sort_values("R2", ascending=False)
display(baseline_table)

## 5. Bio-inspired AI: Genetic Algorithm for feature selection

In [ ]:
selector = GeneticFeatureSelector(
    population_size=20,
    generations=10,
    mutation_rate=0.1,
    crossover_rate=0.8,
    random_state=42,
    model_n_estimators=60,
    max_train_rows=50000,
)

ga_result = selector.fit(X_train, y_train)

print("Best individual:", ga_result.best_individual)
print("Best features:", ga_result.best_features)
print("Best fitness:", ga_result.best_fitness)

history_df = pd.DataFrame(ga_result.history)
display(history_df)

## 6. Final model

In [ ]:
selected_features = ga_result.best_features

final_pipeline = Pipeline([
    ("preprocessor", build_preprocessor(selected_features)),
    ("model", RandomForestRegressor(
        n_estimators=180,
        random_state=42,
        n_jobs=-1,
        max_depth=24,
    )),
])

final_pipeline.fit(X_train[selected_features], y_train)
final_predictions = final_pipeline.predict(X_test[selected_features])
final_metrics = regression_metrics(y_test, final_predictions)

comparison = pd.DataFrame({
    f"Best baseline ({baseline_table.index[0]})": baseline_results[baseline_table.index[0]],
    "GA-selected final model": final_metrics,
}).T

display(comparison)

## 7. Explainability

In [ ]:
model = final_pipeline.named_steps["model"]
preprocessor = final_pipeline.named_steps["preprocessor"]
transformed_names = get_transformed_feature_names(preprocessor)

importance_df = pd.DataFrame({
    "feature": transformed_names,
    "importance": model.feature_importances_,
}).sort_values("importance", ascending=False)

display(importance_df.head(15))

plt.figure(figsize=(10, 6))
sns.barplot(
    data=importance_df.head(15),
    x="importance",
    y="feature",
)
plt.title("Top 15 transformed feature importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()

aggregated = []
for original in selected_features:
    mask = (importance_df["feature"] == original) | (importance_df["feature"].str.startswith(f"{original}_"))
    aggregated.append({
        "feature_group": original,
        "importance": importance_df.loc[mask, "importance"].sum(),
    })

aggregated_df = pd.DataFrame(aggregated).sort_values("importance", ascending=False)
display(aggregated_df)

print("Explanation: tree-based feature importance estimates which transformed input variables contributed most to salary prediction.")

## 8. Save model

In [ ]:
import joblib
import json

os.makedirs("../models", exist_ok=True)

model_package = {
    "selected_features": selected_features,
    "pipeline": final_pipeline,
    "metadata": {
        "target_column": "salary",
        "all_features": ALL_FEATURES,
        "ga_best_fitness": ga_result.best_fitness,
        "ga_history": ga_result.history,
        "final_metrics": final_metrics,
        "best_baseline_name": baseline_table.index[0],
        "best_baseline_metrics": baseline_results[baseline_table.index[0]],
    },
}

joblib.dump(model_package, "../models/salary_model.pkl")
print("Saved model to ../models/salary_model.pkl")

## 9. Notes for Streamlit MVP app

The Streamlit interface is implemented in `app.py`. It loads `models/salary_model.pkl`, displays input widgets, keeps GA-selected features and predicts salary.